In [18]:
import sys
import os

# Proje kökünü python path'e ekleme
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.append(project_root)

print("PYTHONPATH güncellendi:", project_root)


PYTHONPATH güncellendi: c:\Users\emirh\Desktop\buu_llm_rag


In [19]:
pdf_yonetmelik = "../data/raw/BUU_yuksek_ogretim_yonetmeligi.pdf"
pdf_kanun = "../data/raw/yuksek_ogretim_kanunu.pdf"

print(os.path.exists(pdf_yonetmelik), os.path.exists(pdf_kanun))


True True


In [20]:
from src.preprocessing.pdf_loader import load_pdf_pages
from src.preprocessing.text_cleaner import clean_document_as_single_text
from src.preprocessing.chunker import (
    build_chunks_from_pdf_pages,
    split_text_into_articles
)


In [21]:
pages = load_pdf_pages(pdf_kanun)
len(pages)


142

In [22]:
pages[0].text[:500]


'YÜKSEKÖĞRETİM KANUNU\nKanun Numarası : 2547\nKabul Tarihi : 4/11/1981\nYayımlandığı Resmî Gazete : Tarih : 6/11/1981 Sayı : 17506\nYayımlandığı Düstur : Tertip : 5 Cilt : 21 Sayfa : 3\nBİRİNCİ BÖLÜM\nKanunun Amacı, Kapsamı ve Tanımlar\nAmaç:\nMadde 1 – Bu kanunun amacı; yükseköğretimle ilgili amaç ve ilkeleri belirlemek ve\nbütün yükseköğretim kurumlarının ve üst kuruluşlarının teşkilatlanma, işleyiş, görev, yetki ve\nsorumlulukları ile eğitim - öğretim, araştırma, yayım, öğretim elemanları, öğrenciler ve'

In [23]:
cleaned_text = clean_document_as_single_text(pages)
cleaned_text[:1000]


'YÜKSEKÖĞRETİM KANUNU\nKanun Numarası : 2547\nKabul Tarihi : 4/11/1981\nYayımlandığı Resmî Gazete : Tarih : 6/11/1981 Sayı : 17506\nYayımlandığı Düstur : Tertip : 5 Cilt : 21 Sayfa : 3\nBİRİNCİ BÖLÜM\nKanunun Amacı, Kapsamı ve Tanımlar\nAmaç:\nMadde 1 – Bu kanunun amacı; yükseköğretimle ilgili amaç ve ilkeleri belirlemek ve bütün yükseköğretim kurumlarının ve üst kuruluşlarının teşkilatlanma, işleyiş, görev, yetki ve sorumlulukları ile eğitim - öğretim, araştırma, yayım, öğretim elemanları, öğrenciler ve diğer personel ile ilgili esasları bir bütünlük içinde düzenlemektir.\nKapsam:\nMadde 2 – Bu kanun; yükseköğretim üst kuruluşlarını, bütün yükseköğretim kurumlarını, bağlı birimlerini ve bunlarla ilgili faaliyet ve esasları kapsar.\n(Değişik ikinci fıkra: 15/8/2017-KHK-694/44 md.; Aynen kabul: 1/2/2018-7078/41 md.) Milli Savunma Bakanlığı ve İçişleri Bakanlığına bağlı yükseköğretim kurumlarıyla ilgili özel kanun hükümleri saklıdır.\nTanımlar:\nMadde 3 – (Değişik: 17/8/1983 - 2880/1 md.

In [24]:
articles = split_text_into_articles(cleaned_text)
len(articles)


217

In [25]:
articles[0].article_no, articles[0].text[:800]


('MADDE 1',
 'Madde 1 – Bu kanunun amacı; yükseköğretimle ilgili amaç ve ilkeleri belirlemek ve bütün yükseköğretim kurumlarının ve üst kuruluşlarının teşkilatlanma, işleyiş, görev, yetki ve sorumlulukları ile eğitim - öğretim, araştırma, yayım, öğretim elemanları, öğrenciler ve diğer personel ile ilgili esasları bir bütünlük içinde düzenlemektir.\nKapsam:')

In [26]:
chunks = build_chunks_from_pdf_pages(
    pages=pages,
    doc_id="yuksek_ogretim_kanunu",
    doc_type="kanun",
    doc_name="2547 Sayılı Yükseköğretim Kanunu"
)

len(chunks)


270

In [27]:
for c in chunks[:3]:
    print("ID:", c.id)
    print("Madde:", c.article_no)
    print("Paragraph:", c.paragraph_no)
    print("Text:", c.text[:300])
    print("----\n")


ID: yuksek_ogretim_kanunu_madde1
Madde: MADDE 1
Paragraph: None
Text: Madde 1 – Bu kanunun amacı; yükseköğretimle ilgili amaç ve ilkeleri belirlemek ve bütün yükseköğretim kurumlarının ve üst kuruluşlarının teşkilatlanma, işleyiş, görev, yetki ve sorumlulukları ile eğitim - öğretim, araştırma, yayım, öğretim elemanları, öğrenciler ve diğer personel ile ilgili esasları
----

ID: yuksek_ogretim_kanunu_madde2
Madde: MADDE 2
Paragraph: None
Text: Madde 2 – Bu kanun; yükseköğretim üst kuruluşlarını, bütün yükseköğretim kurumlarını, bağlı birimlerini ve bunlarla ilgili faaliyet ve esasları kapsar.
(Değişik ikinci fıkra: 15/8/2017-KHK-694/44 md.; Aynen kabul: 1/2/2018-7078/41 md.) Milli Savunma Bakanlığı ve İçişleri Bakanlığına bağlı yükseköğre
----

ID: yuksek_ogretim_kanunu_madde3_1
Madde: MADDE 3
Paragraph: None
Text: Madde 3 – (Değişik: 17/8/1983 - 2880/1 md.)
Bu Kanunda geçen kavram ve terimlerin tanımları aşağıda belirtilmiştir.
a) Yükseköğretim: Milli eğitim sistemi içinde, ortaöğretim

In [28]:
from src.preprocessing.chunker import chunks_to_dicts
import json

output_path = "../data/processed/yuksek_ogretim_kanunu_chunks.jsonl"

with open(output_path, "w", encoding="utf-8") as f:
    for d in chunks_to_dicts(chunks):
        f.write(json.dumps(d, ensure_ascii=False) + "\n")

print("Kaydedildi:", output_path)


Kaydedildi: ../data/processed/yuksek_ogretim_kanunu_chunks.jsonl


In [29]:
from src.preprocessing.pdf_loader import load_pdf_pages
from src.preprocessing.text_cleaner import clean_document_as_single_text
from src.preprocessing.chunker import build_chunks_from_pdf_pages, chunks_to_dicts

import json
import os

pdf_yonetmelik = "../data/raw/BUU_yuksek_ogretim_yonetmeligi.pdf"
print(os.path.exists(pdf_yonetmelik))


True


In [30]:
# 1) Sayfaları yükle
yon_pages = load_pdf_pages(pdf_yonetmelik)
len(yon_pages), yon_pages[0].text[:400]


(13,
 '28 Haziran 2020 PAZAR Resmî Gazete Sayı : 31169\nYÖNETMELİK\nBursa Uludağ Üniversitesinden:\nBURSA ULUDAĞ ÜNİVERSİTESİ LİSANSÜSTÜ EĞİTİM\nVE ÖĞRETİM YÖNETMELİĞİ\nBİRİNCİ BÖLÜM\nAmaç, Kapsam, Dayanak ve Tanımlar\nAmaç\nMADDE 1 – (1) Bu Yönetmeliğin amacı; Bursa Uludağ Üniversitesine bağlı enstitülerde yürütülen lisansüstü\neğitim-öğretim programlarına ilişkin usul ve esasları düzenlemektir.\nKapsam\nMADDE 2 –')

In [31]:
cleaned_yon_text = clean_document_as_single_text(yon_pages)
cleaned_yon_text[:1000]


'28 Haziran 2020 PAZAR Resmî Gazete Sayı : 31169\nYÖNETMELİK\nBursa Uludağ Üniversitesinden:\nBURSA ULUDAĞ ÜNİVERSİTESİ LİSANSÜSTÜ EĞİTİM\nVE ÖĞRETİM YÖNETMELİĞİ\nBİRİNCİ BÖLÜM\nAmaç, Kapsam, Dayanak ve Tanımlar\nAmaç\nMADDE 1 – (1) Bu Yönetmeliğin amacı; Bursa Uludağ Üniversitesine bağlı enstitülerde yürütülen lisansüstü eğitim-öğretim programlarına ilişkin usul ve esasları düzenlemektir.\nKapsam\nMADDE 2 – (1) Bu Yönetmelik; Bursa Uludağ Üniversitesine bağlı enstitülerde yürütülen tezli ve tezsiz yüksek lisans, doktora ve sanat dallarında yapılan sanatta yeterlik ve uzaktan öğretim ile güz, bahar ve yaz dönemlerinde yürütülen lisansüstü programlarından oluşan lisansüstü eğitim-öğretim ve sınavlara ilişkin hükümleri kapsar.\nDayanak\nMADDE 3 – (1) Bu Yönetmelik, 4/11/1981 tarihli ve 2547 sayılı Yükseköğretim Kanununun 14 üncü ve 44 üncü maddelerine dayanılarak hazırlanmıştır.\nTanımlar\nMADDE 4 – (1) Bu Yönetmelikte geçen;\na) Akademik takvim: Lisansüstü eğitimlerde yarıyıl, yaz dönem

In [32]:
yon_chunks = build_chunks_from_pdf_pages(
    pages=yon_pages,
    doc_id="buu_yonetmelik",
    doc_type="yönetmelik",
    doc_name="Bursa Uludağ Üniversitesi Lisansüstü Eğitim-Öğretim Yönetmeliği"
)

len(yon_chunks)


74

In [33]:
for c in yon_chunks[:5]:
    print("ID:", c.id)
    print("Madde:", c.article_no)
    print("Paragraph:", c.paragraph_no)
    print("Text:", c.text[:300])
    print("----\n")


ID: buu_yonetmelik_madde1
Madde: MADDE 1
Paragraph: None
Text: MADDE 1 – (1) Bu Yönetmeliğin amacı; Bursa Uludağ Üniversitesine bağlı enstitülerde yürütülen lisansüstü eğitim-öğretim programlarına ilişkin usul ve esasları düzenlemektir.
Kapsam
----

ID: buu_yonetmelik_madde2
Madde: MADDE 2
Paragraph: None
Text: MADDE 2 – (1) Bu Yönetmelik; Bursa Uludağ Üniversitesine bağlı enstitülerde yürütülen tezli ve tezsiz yüksek lisans, doktora ve sanat dallarında yapılan sanatta yeterlik ve uzaktan öğretim ile güz, bahar ve yaz dönemlerinde yürütülen lisansüstü programlarından oluşan lisansüstü eğitim-öğretim ve sın
----

ID: buu_yonetmelik_madde3
Madde: MADDE 3
Paragraph: None
Text: MADDE 3 – (1) Bu Yönetmelik, 4/11/1981 tarihli ve 2547 sayılı Yükseköğretim Kanununun 14 üncü ve 44 üncü maddelerine dayanılarak hazırlanmıştır.
Tanımlar
----

ID: buu_yonetmelik_madde4_1
Madde: MADDE 4
Paragraph: None
Text: MADDE 4 –
----

ID: buu_yonetmelik_madde4_2
Madde: MADDE 4
Paragraph: (1)
Text: (1) Bu Yönet

In [34]:
yon_output = "../data/processed/buu_yonetmelik_chunks.jsonl"

with open(yon_output, "w", encoding="utf-8") as f:
    for d in chunks_to_dicts(yon_chunks):
        f.write(json.dumps(d, ensure_ascii=False) + "\n")

print("Kaydedildi:", yon_output)


Kaydedildi: ../data/processed/buu_yonetmelik_chunks.jsonl
